# 03 — Germany: Autobahn AnalysisIdentify motorway accidents from Unfallatlas via spatial join with OSM,then split by unlimited vs limited speed sections.

In [ ]:
from pathlib import Pathimport pandas as pdimport numpy as npimport geopandas as gpdimport osmnx as oximport matplotlib.pyplot as pltimport seaborn as snsfrom shapely.geometry import Pointimport sys, warningswarnings.filterwarnings('ignore')sys.path.insert(0, '../src')from autobahn_safety.data_loaders import load_unfallatlas, load_destatis_autobahnfrom autobahn_safety.preprocessing import clean_unfallatlasfrom autobahn_safety.analysis import compute_accident_rate, severity_indexfrom autobahn_safety.visualization import plot_rate_trend, plot_rate_comparisonDATA_RAW = Path('../data/raw')DATA_PROC = Path('../data/processed')DATA_PROC.mkdir(parents=True, exist_ok=True)sns.set_theme(style='whitegrid')print('Ready.')

## 1. Load and clean Unfallatlas

In [ ]:
df_ua = load_unfallatlas(DATA_RAW / 'germany' / 'unfallatlas', list(range(2016, 2025)))df_ua = clean_unfallatlas(df_ua)print(f'{len(df_ua):,} accident records, {df_ua["year"].nunique()} years')

## 2. Download German Autobahn network from OSMWe use OSMnx to fetch the motorway network for Germany.This includes `maxspeed` tags which tell us whether a section has a speed limit.**Sections with no speed limit (or `maxspeed=DE:motorway`) = unlimited speed (Tempolimit 0).**⚠️ This download takes a few minutes and requires internet access.  The result is cached to `data/processed/` for subsequent runs.

In [ ]:
MOTORWAY_CACHE = DATA_PROC / 'germany_motorways.gpkg'if MOTORWAY_CACHE.exists():    print('Loading cached motorway network...')    gdf_motorways = gpd.read_file(MOTORWAY_CACHE)    print(f'Loaded {len(gdf_motorways):,} motorway segments')else:    print('Downloading German motorway network from OSM (may take a few minutes)...')    # Query OSM for motorway edges in Germany    G = ox.graph_from_place(        'Germany',        network_type='drive',        custom_filter='["highway"~"motorway|motorway_link"]',        retain_all=False,    )    _, edges = ox.graph_to_gdfs(G)    gdf_motorways = edges[['geometry', 'highway', 'maxspeed', 'name', 'ref']].copy()    gdf_motorways = gdf_motorways.reset_index(drop=True)    gdf_motorways.to_file(MOTORWAY_CACHE, driver='GPKG')    print(f'Downloaded and cached {len(gdf_motorways):,} motorway segments')gdf_motorways.head(3)

In [ ]:
# Classify sections: unlimited vs speed-limited# In Germany, motorways without an explicit speed limit are "unlimited"# maxspeed=None or "DE:motorway" or "de" => unlimiteddef is_unlimited(maxspeed):    if pd.isna(maxspeed):        return True  # No posted limit = unlimited    ms = str(maxspeed).lower().strip()    return ms in ('', 'de:motorway', 'de', 'none', 'signals')gdf_motorways['unlimited'] = gdf_motorways['maxspeed'].apply(is_unlimited)print(f'Unlimited sections : {gdf_motorways["unlimited"].sum():,}')print(f'Speed-limited      : {(~gdf_motorways["unlimited"]).sum():,}')print(f'Fraction unlimited : {gdf_motorways["unlimited"].mean()*100:.1f}%')

## 3. Spatial join: classify accidents as motorway / non-motorway

In [ ]:
# Convert accident points to GeoDataFrame (WGS84)df_clean = df_ua.dropna(subset=['lon', 'lat'])gdf_acc = gpd.GeoDataFrame(    df_clean,    geometry=gpd.points_from_xy(df_clean['lon'], df_clean['lat']),    crs='EPSG:4326')# Project both to EPSG:25832 (Germany UTM) for accurate distance buffergdf_acc = gdf_acc.to_crs('EPSG:25832')gdf_mw  = gdf_motorways.to_crs('EPSG:25832')# Buffer motorway lines by 50mprint('Buffering motorway network (50m)...')gdf_mw_buf = gdf_mw.copy()gdf_mw_buf['geometry'] = gdf_mw_buf.geometry.buffer(50)print('Spatial join...')joined = gpd.sjoin(gdf_acc, gdf_mw_buf[['geometry', 'unlimited']], how='left', predicate='within')# An accident can match multiple buffers — take the first match per accidentjoined = joined[~joined.index.duplicated(keep='first')]df_ua['on_motorway'] = ~joined['unlimited'].isna()df_ua['on_unlimited'] = joined['unlimited'].fillna(False)df_ua['on_limited_mw'] = df_ua['on_motorway'] & ~df_ua['on_unlimited']print(f'\nOn motorway      : {df_ua["on_motorway"].sum():,} ({df_ua["on_motorway"].mean()*100:.1f}%)')print(f'On unlimited     : {df_ua["on_unlimited"].sum():,} ({df_ua["on_unlimited"].mean()*100:.1f}%)')print(f'On limited mw    : {df_ua["on_limited_mw"].sum():,} ({df_ua["on_limited_mw"].mean()*100:.1f}%)')

In [ ]:
# Save classified accidentsout = DATA_PROC / 'unfallatlas_classified.parquet'df_ua.to_parquet(out, index=False)print(f'Saved classified accidents to {out}')

## 4. Accident trends: unlimited vs limited Autobahn

In [ ]:
# Annual counts by groupgroups = {    'Unlimited Autobahn': df_ua[df_ua['on_unlimited']],    'Speed-limited Autobahn': df_ua[df_ua['on_limited_mw']],    'Non-motorway': df_ua[~df_ua['on_motorway']],}annual = []for label, grp in groups.items():    by_year = grp.groupby('year').agg(        total=('severity', 'count'),        fatal=('severity', lambda x: (x == 'fatal').sum()),        serious=('severity', lambda x: (x == 'serious_injury').sum()),    ).reset_index()    by_year['group'] = label    annual.append(by_year)df_annual = pd.concat(annual, ignore_index=True)# Plot accident countsfig, ax = plt.subplots(figsize=(11, 5))for label, grp in df_annual.groupby('group'):    ax.plot(grp['year'], grp['total'], marker='o', label=label, linewidth=2)ax.set_title('Accident counts by road type (Unfallatlas 2016–2024)')ax.set_xlabel('Year'); ax.set_ylabel('Accidents')ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Severity index (fatalities per accident)df_annual['severity_idx'] = df_annual['fatal'] / df_annual['total']fig, ax = plt.subplots(figsize=(11, 5))for label, grp in df_annual.groupby('group'):    ax.plot(grp['year'], grp['severity_idx'], marker='o', label=label, linewidth=2)ax.set_title('Severity index (fatalities / accident) by road type')ax.set_xlabel('Year'); ax.set_ylabel('Severity index')ax.legend(); plt.tight_layout(); plt.show()print('\nMean severity index 2016-2024:')print(df_annual.groupby('group')['severity_idx'].mean().sort_values(ascending=False).to_string())

## 5. Long-term Autobahn trend (Destatis 1979–2021)

In [ ]:
df_de = load_destatis_autobahn(    DATA_RAW / 'germany' / 'destatis' / 'destatis_verkehrsunfaelle_zeitreihen.xlsx')fig, axes = plt.subplots(1, 2, figsize=(13, 5))axes[0].plot(df_de['year'], df_de['accidents_personal_injury'], color='steelblue', linewidth=2)axes[0].fill_between(df_de['year'], df_de['accidents_personal_injury'], alpha=0.2, color='steelblue')axes[0].set_title('Autobahn: accidents with personal injury (1979–2021)')axes[0].set_xlabel('Year'); axes[0].set_ylabel('Accidents')axes[1].plot(df_de['year'], df_de['accidents_fatal'], color='crimson', linewidth=2)axes[1].fill_between(df_de['year'], df_de['accidents_fatal'], alpha=0.2, color='crimson')axes[1].set_title('Autobahn: fatal accidents (1979–2021)')axes[1].set_xlabel('Year'); axes[1].set_ylabel('Fatal accidents')plt.suptitle('German Autobahn — long-term safety trend', fontweight='bold')plt.tight_layout(); plt.show()